# ZelusBench: Selective Attention — Signal in Noise

Can the model compute through a relevant dependency chain when surrounded by irrelevant points?

**Independent variable**: noise multiplier (num_points / chain_depth)
- Chain depth fixed at 5
- Noise multipliers: [1.0, 1.5, 2.0, 3.0, 4.0] → num_points = [5, 8, 10, 15, 20]
- At 1.0x every point is on the critical path; at 4.0x, 75% are distractors
- POSITION queries only, targeting deep points
- transform_prob=0.1, dim=3 fixed
- 10 seeds per noise level = 50 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTH = 5
NOISE_LEVELS = [1.0, 1.5, 2.0, 3.0, 4.0]  # multiplier on depth
SEEDS = 10
TOTAL = len(NOISE_LEVELS) * SEEDS
print(f"ZelusBench Selective Attention — Signal in Noise")
print(f"Depth: {DEPTH} | Noise levels: {NOISE_LEVELS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_selective")
def zelusbench_selective(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scenario_avgs = []
    level_scores = {}
    scenario_num = 0
    for noise in NOISE_LEVELS:
        num_pts = round(DEPTH * noise)
        level_scores[noise] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "dim": 3,
                "min_chain_depth": DEPTH, "max_chain_depth": DEPTH,
                "num_points": num_pts,
                "transform_prob": 0.1,
                "query_types": [QueryType.POSITION],
                "query_min_depth": max(1, DEPTH - 2),
                "num_queries": 3,
                "seed": int(noise * 1000) + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"selective_{noise}_{i}")
            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            avg = float(np.mean([s.score for s in scores]))
            level_scores[noise].append(avg)
            all_scenario_avgs.append(avg)
            print(f"  [{scenario_num}/{TOTAL}] noise={noise}x pts={num_pts} avg={avg:.2f} | lb={cfg.leaf_bias}")
        print(f"  >> Noise {noise}x mean: {float(np.mean(level_scores[noise])):.3f}")
    print("\n" + "=" * 60)
    for noise in NOISE_LEVELS:
        num_pts = round(DEPTH * noise)
        avg = round(float(np.mean(level_scores[noise])), 3)
        sem = round(float(np.std(level_scores[noise]) / np.sqrt(len(level_scores[noise]))), 3)
        print(f"  noise={noise}x  pts={num_pts:2d}  accuracy={avg:.3f} +/- {sem:.3f}")
        kbench.assertions.assert_true(
            avg > 0.0,
            expectation=f"Selective noise={noise}x ({num_pts} pts): accuracy={avg:.3f} +/- {sem:.3f}"
        )
    score = round(float(np.mean(all_scenario_avgs)), 3)
    confidence = round(float(np.std(all_scenario_avgs) / np.sqrt(len(all_scenario_avgs))), 3)
    print(f"\nScore: {score:.3f} +/- {confidence:.3f} (SEM) | {len(all_scenario_avgs)} scenarios | {time.time()-_start:.1f}s")
    return score, confidence

zelusbench_selective.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_selective